# Example: Nearest-Kpoint Approach (InAs)

This notebook demonstrates the `nearest_kpoint` approach for computing Auger recombination coefficients using InAs VASP data from the `test-files/` directory.

The `nearest_kpoint` approach maps the momentum-conserving k-vector **k₄** to the closest k-point in the existing SCF grid. This requires `ISYM = -1` in the VASP INCAR so that the full BZ k-mesh is available.

---

In [1]:
import sys, os

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd())))

from auger import AugerCalculator, utilities
import numpy as np

## Configuration

In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────
VASP_FOLDER = "../test-files/InAs/nearest-kpoint-scf-10"
RESULTS_DIR = "./results/InAs/nearest_kpoint"

# ── Physical parameters ────────────────────────────────────────────────────
TEMPERATURE    = 300                   # K
DOPING         = 0                     # cm⁻³ (intrinsic)
EXCESS_CARRIER = 1e17                  # cm⁻³

# ── Material parameters (InAs) ─────────────────────────────────────────────
DIELECTRIC     = 15.15                 # Static dielectric constant
BAND_GAP       = 0.4                   # Experimental band gap (eV)

# ── Energy windows ─────────────────────────────────────────────────────────
CB_WINDOW      = 0.3                   # eV above CBM
VB_WINDOW      = 0.3                   # eV below VBM

---
## Step 0 — Create calculator and assign bands

InAs has a narrow DFT band gap that pymatgen may misidentify. We manually assign the band indices.

In [3]:
calc = AugerCalculator(T=TEMPERATURE, nd=DOPING)

# InAs: first CB is band 9, last VB is band 8 (0-based)
calc.assign_firstCB_and_lastVB(firstCB_index=9, lastVB_index=8)


                              AUGER RECOMBINATION CALCULATOR                              
│ Temperature:                   300 K                                                   │
│ Doping Concentration:          0.00e+00 cm⁻³                                           │
│ Doping Type:                   intrinsic                                               │

  First CB index: 9  |  Last VB index: 8


---
## Step 1 — Parse VASP data

Parse the SCF output files. The `force_gap` parameter corrects the DFT band gap to the experimental value.

In [5]:
calc.parse_BS_data(
    folder_path=VASP_FOLDER,
    write_path=RESULTS_DIR,
    force_gap=BAND_GAP,
)


──────────────────────────────────────────────────────────────────────────────────────────
📂 Parsing VASP data from: ../test-files/InAs/nearest-kpoint-scf-10
──────────────────────────────────────────────────────────────────────────────────────────
  Material: InAs  |  Band gap: 0.0 eV
  Applying scissor shift: +0.4000 eV
  Saved to: ./results/InAs/nearest_kpoint/
──────────────────────────────────────────────────────────────────────────────────────────



---
## Step 2 — Import parsed data

In [6]:
calc.import_parsed_BS_data(from_folder=RESULTS_DIR)


──────────────────────────────────────────────────────────────────────────────────────────
📥 Importing from: ./results/InAs/nearest_kpoint
──────────────────────────────────────────────────────────────────────────────────────────
  Material: InAs
  K-grid:   (10, 10, 10)  (1000 irreducible k-points)
  Bands:    16  |  Gap: 0.4000 eV
──────────────────────────────────────────────────────────────────────────────────────────



---
## Step 3 — Carrier concentrations

In [7]:
fn, fp = calc.calculate_carrier_concentrations(delta_n=EXCESS_CARRIER)

print(f"n  = {calc.n:.4e} cm⁻³")
print(f"p  = {calc.p:.4e} cm⁻³")
print(f"ni = {calc.ni:.4e} cm⁻³")


──────────────────────────────────────────────────────────────────────────────────────────
⚙  Calculating carrier concentrations …
──────────────────────────────────────────────────────────────────────────────────────────

  NON-EQUILIBRIUM CARRIER CONCENTRATIONS
    n = 1.2279e+17 cm⁻³   |   p = 1.2279e+17 cm⁻³
    Efn = 0.2537 eV    |   Efp = 0.1665 eV
    ni  = 2.2790e+16 cm⁻³
  ⏱  00h 00m 05s
──────────────────────────────────────────────────────────────────────────────────────────

n  = 1.2279e+17 cm⁻³
p  = 1.2279e+17 cm⁻³
ni = 2.2790e+16 cm⁻³


---
## Step 4 — Energy-window selection (optional)

In [8]:
CB_auto, VB_auto = calc.calculate_energy_cutoffs(charge_threshold=0.99)

print(f"Auto CB_window = {CB_auto:.4f} eV")
print(f"Auto VB_window = {VB_auto:.4f} eV")

  Energy windows (99% of carriers):
    CB: 0.4000 → 0.4088 eV  (width 0.0088 eV)
    VB: -0.1696 → 0.0000 eV  (width 0.1696 eV)
Auto CB_window = 0.0088 eV
Auto VB_window = 0.1696 eV


---
## Step 5 — Pair generation (EEH)

Using the `nearest_kpoint` approach with the `Max_Heap` search mode.

In [9]:
gen_eeh = calc.create_auger_pairs(
    CB_window=CB_WINDOW,
    VB_window=VB_WINDOW,
    auger_type="eeh",
    approach="nearest_kpoint",
    search_mode="Brute_Force",
    num_top_pairs='all',
)


                                  AUGER PAIR GENERATION                                   
  Type: EEH  |  Approach: nearest_kpoint  |  Search: Brute_Force
  CB window: 0.3000 eV  |  VB window: 0.3000 eV
──────────────────────────────────────────────────────────────────────────────────────────

  Initializing energy states …
    Type: EEH  |  Approach: nearest_kpoint  |  Search: Brute_Force
    E1(e): 1  |  E3(e): 1  |  E4(h): 31  |  combos: 31

  Pair creation complete: 31 pairs
  Saved 31 pairs → auger_eeh_pairs_1000_*.csv

  ✓ 31 pairs  |  ⏱  00h 00m 00s



---
## Step 6 — Matrix elements (EEH)

In [10]:
me_eeh = calc.calculate_matrix_elements(
    auger_type="eeh",
    wavecar_files=VASP_FOLDER + "/WAVECAR",
    dielectric_constant=DIELECTRIC,
    num_matrix_elements="all",
)

  Inverse Debye length: 0.010652 Å⁻¹

  Computing 31 matrix elements on 15 cores …
    [    31/31] 100.0%  avg |M|²=1.6166e+00 eV²
  Saved → ./results/InAs/nearest_kpoint\eeh_matrix_elements_1000.jsonl
  ⏱  00h 02m 09s


---
## Step 7 — Auger coefficients (EEH)

In [11]:
results_eeh = calc.calculate_auger_rates(
    auger_type="eeh",
    delta_function=("Gaussian", "Lorentzian", "Rectangular"),
    FWHM=(0.001, 0.005, 0.01, 0.03, 0.05, 0.07, 0.1, 0.15, 0.2),
)


                              AUGER COEFFICIENT CALCULATION                               
  Type: EEH  |  T = 300 K
──────────────────────────────────────────────────────────────────────────────────────────

  ✓ Saved Auger_coefficients_eeh_1000.csv
    Gaussian (FWHM=0.05):  C = 6.3514e-98 cm⁶/s
    Lorentzian (FWHM=0.05):  C = 7.6045e-32 cm⁶/s
    Rectangular (FWHM=0.05):  C = 0.0000e+00 cm⁶/s
  ⏱  00h 00m 00s



In [12]:
import pandas as pd

df_eeh = pd.DataFrame(results_eeh)
print("EEH Auger Coefficients (C_n) for InAs")
print("=" * 60)
for delta in df_eeh["Delta function"].unique():
    print(f"\n  {delta}:")
    sub = df_eeh[df_eeh["Delta function"] == delta]
    for _, row in sub.iterrows():
        print(f"    FWHM = {row['FWHM']:.3f} eV  ->  C_n = {row['Auger coefficient']:.4e} cm^6/s")

EEH Auger Coefficients (C_n) for InAs

  Gaussian:
    FWHM = 0.001 eV  ->  C_n = 0.0000e+00 cm^6/s
    FWHM = 0.005 eV  ->  C_n = 0.0000e+00 cm^6/s
    FWHM = 0.010 eV  ->  C_n = 0.0000e+00 cm^6/s
    FWHM = 0.030 eV  ->  C_n = 1.1655e-219 cm^6/s
    FWHM = 0.050 eV  ->  C_n = 6.3514e-98 cm^6/s
    FWHM = 0.070 eV  ->  C_n = 1.8091e-64 cm^6/s
    FWHM = 0.100 eV  ->  C_n = 8.9736e-47 cm^6/s
    FWHM = 0.150 eV  ->  C_n = 2.0183e-37 cm^6/s
    FWHM = 0.200 eV  ->  C_n = 3.2739e-34 cm^6/s

  Lorentzian:
    FWHM = 0.001 eV  ->  C_n = 1.5275e-33 cm^6/s
    FWHM = 0.005 eV  ->  C_n = 7.6372e-33 cm^6/s
    FWHM = 0.010 eV  ->  C_n = 1.5272e-32 cm^6/s
    FWHM = 0.030 eV  ->  C_n = 4.5754e-32 cm^6/s
    FWHM = 0.050 eV  ->  C_n = 7.6045e-32 cm^6/s
    FWHM = 0.070 eV  ->  C_n = 1.0602e-31 cm^6/s
    FWHM = 0.100 eV  ->  C_n = 1.5014e-31 cm^6/s
    FWHM = 0.150 eV  ->  C_n = 2.2049e-31 cm^6/s
    FWHM = 0.200 eV  ->  C_n = 2.8563e-31 cm^6/s

  Rectangular:
    FWHM = 0.001 eV  ->  C_n = 0.00

---
## Repeat for EHH ($C_p$)

In [13]:
gen_ehh = calc.create_auger_pairs(
    CB_window=CB_WINDOW,
    VB_window=VB_WINDOW,
    auger_type="ehh",
    approach="nearest_kpoint",
    search_mode="Brute_Force",
    num_top_pairs='all',
)


                                  AUGER PAIR GENERATION                                   
  Type: EHH  |  Approach: nearest_kpoint  |  Search: Brute_Force
  CB window: 0.3000 eV  |  VB window: 0.3000 eV
──────────────────────────────────────────────────────────────────────────────────────────

  Initializing energy states …
    Type: EHH  |  Approach: nearest_kpoint  |  Search: Brute_Force
    E1(e): 1  |  E2(h): 31  |  E3(h): 31  |  combos: 961

  Pair creation complete: 961 pairs
  Saved 961 pairs → auger_ehh_pairs_1000_*.csv

  ✓ 961 pairs  |  ⏱  00h 00m 00s



In [14]:
me_ehh = calc.calculate_matrix_elements(
    auger_type="ehh",
    wavecar_files=VASP_FOLDER + "/WAVECAR",
    dielectric_constant=DIELECTRIC,
    num_matrix_elements="all",
)

  Inverse Debye length: 0.010652 Å⁻¹

  Computing 961 matrix elements on 15 cores …
    [   961/961] 100.0%  avg |M|²=3.0543e+00 eV²
  Saved → ./results/InAs/nearest_kpoint\ehh_matrix_elements_1000.jsonl
  ⏱  00h 56m 06s


In [15]:
results_ehh = calc.calculate_auger_rates(
    auger_type="ehh",
    delta_function=("Gaussian", "Lorentzian", "Rectangular"),
    FWHM=(0.001, 0.005, 0.01, 0.03, 0.05, 0.07, 0.1, 0.15, 0.2),
)


                              AUGER COEFFICIENT CALCULATION                               
  Type: EHH  |  T = 300 K
──────────────────────────────────────────────────────────────────────────────────────────

  ✓ Saved Auger_coefficients_ehh_1000.csv
    Gaussian (FWHM=0.05):  C = 1.7902e-30 cm⁶/s
    Lorentzian (FWHM=0.05):  C = 2.6124e-30 cm⁶/s
    Rectangular (FWHM=0.05):  C = 1.8964e-30 cm⁶/s
  ⏱  00h 00m 00s



In [16]:
df_ehh = pd.DataFrame(results_ehh)
print("EHH Auger Coefficients (C_p) for InAs")
print("=" * 60)
for delta in df_ehh["Delta function"].unique():
    print(f"\n  {delta}:")
    sub = df_ehh[df_ehh["Delta function"] == delta]
    for _, row in sub.iterrows():
        print(f"    FWHM = {row['FWHM']:.3f} eV  ->  C_p = {row['Auger coefficient']:.4e} cm^6/s")

EHH Auger Coefficients (C_p) for InAs

  Gaussian:
    FWHM = 0.001 eV  ->  C_p = 6.8807e-64 cm^6/s
    FWHM = 0.005 eV  ->  C_p = 7.0196e-31 cm^6/s
    FWHM = 0.010 eV  ->  C_p = 3.9687e-30 cm^6/s
    FWHM = 0.030 eV  ->  C_p = 2.7267e-30 cm^6/s
    FWHM = 0.050 eV  ->  C_p = 1.7902e-30 cm^6/s
    FWHM = 0.070 eV  ->  C_p = 1.4677e-30 cm^6/s
    FWHM = 0.100 eV  ->  C_p = 1.4568e-30 cm^6/s
    FWHM = 0.150 eV  ->  C_p = 1.5517e-30 cm^6/s
    FWHM = 0.200 eV  ->  C_p = 1.7626e-30 cm^6/s

  Lorentzian:
    FWHM = 0.001 eV  ->  C_p = 5.4374e-31 cm^6/s
    FWHM = 0.005 eV  ->  C_p = 2.2839e-30 cm^6/s
    FWHM = 0.010 eV  ->  C_p = 3.0918e-30 cm^6/s
    FWHM = 0.030 eV  ->  C_p = 2.6817e-30 cm^6/s
    FWHM = 0.050 eV  ->  C_p = 2.6124e-30 cm^6/s
    FWHM = 0.070 eV  ->  C_p = 2.8177e-30 cm^6/s
    FWHM = 0.100 eV  ->  C_p = 3.2711e-30 cm^6/s
    FWHM = 0.150 eV  ->  C_p = 4.0708e-30 cm^6/s
    FWHM = 0.200 eV  ->  C_p = 4.8052e-30 cm^6/s

  Rectangular:
    FWHM = 0.001 eV  ->  C_p = 0.000

---
## Total Auger coefficient

In [17]:
merged = pd.merge(df_eeh, df_ehh, on=["FWHM", "Delta function"], suffixes=("_eeh", "_ehh"))
merged["C_total"] = merged["Auger coefficient_eeh"] + merged["Auger coefficient_ehh"]

print("Total Auger Coefficient (C_n + C_p) for InAs")
print("=" * 70)
for delta in merged["Delta function"].unique():
    print(f"\n  {delta}:")
    sub = merged[merged["Delta function"] == delta]
    for _, row in sub.iterrows():
        print(f"    FWHM = {row['FWHM']:.3f} eV  ->  C_total = {row['C_total']:.4e} cm^6/s")

Total Auger Coefficient (C_n + C_p) for InAs

  Gaussian:
    FWHM = 0.001 eV  ->  C_total = 6.8807e-64 cm^6/s
    FWHM = 0.005 eV  ->  C_total = 7.0196e-31 cm^6/s
    FWHM = 0.010 eV  ->  C_total = 3.9687e-30 cm^6/s
    FWHM = 0.030 eV  ->  C_total = 2.7267e-30 cm^6/s
    FWHM = 0.050 eV  ->  C_total = 1.7902e-30 cm^6/s
    FWHM = 0.070 eV  ->  C_total = 1.4677e-30 cm^6/s
    FWHM = 0.100 eV  ->  C_total = 1.4568e-30 cm^6/s
    FWHM = 0.150 eV  ->  C_total = 1.5517e-30 cm^6/s
    FWHM = 0.200 eV  ->  C_total = 1.7630e-30 cm^6/s

  Lorentzian:
    FWHM = 0.001 eV  ->  C_total = 5.4527e-31 cm^6/s
    FWHM = 0.005 eV  ->  C_total = 2.2915e-30 cm^6/s
    FWHM = 0.010 eV  ->  C_total = 3.1071e-30 cm^6/s
    FWHM = 0.030 eV  ->  C_total = 2.7275e-30 cm^6/s
    FWHM = 0.050 eV  ->  C_total = 2.6885e-30 cm^6/s
    FWHM = 0.070 eV  ->  C_total = 2.9237e-30 cm^6/s
    FWHM = 0.100 eV  ->  C_total = 3.4213e-30 cm^6/s
    FWHM = 0.150 eV  ->  C_total = 4.2913e-30 cm^6/s
    FWHM = 0.200 eV  ->  C